# ĐỒ ÁN MÔN HỌC MÁY: DỰ ĐOÁN NGUY CƠ TIỂU ĐƯỜNG
## NOTEBOOK 1: PHÂN TÍCH KHÁM PHÁ DỮ LIỆU (EDA) & TIỀN XỬ LÝ CHỐNG DATA LEAKAGE
---
- **Sinh viên thực hiện:** Đồ án Machine Learning cơ bản
- **Bộ dữ liệu:** Pima Indians Diabetes Dataset (Kaggle / UCI)
- **3 Đặc trưng sử dụng:** `Glucose`, `BMI`, `Age`
- **Nhãn mục tiêu:** `Outcome` (0: Không mắc tiểu đường, 1: Có mắc tiểu đường)

### 1. Khai báo thư viện phục vụ phân tích

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Thiết lập phong cách đồ họa chuyên nghiệp
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'sans-serif']
print('Đã nạp toàn bộ thư viện thành công!')

### 2. Đọc dữ liệu và trích xuất 3 thuộc tính đầu vào
Theo yêu cầu đề tài, chỉ sử dụng 3 đặc trưng lâm sàng cơ bản: Glucose, BMI, Age.

In [ ]:
data_path = os.path.join('..', 'data', 'diabetes.csv')
df = pd.read_csv(data_path)

features = ['Glucose', 'BMI', 'Age']
target = 'Outcome'
df_sub = df[features + [target]].copy()

print(f'Kích thước dữ liệu: {df_sub.shape[0]} dòng, {df_sub.shape[1]} cột')
df_sub.head(10)

### 3. Phát hiện giá trị 0 không hợp lý (Hidden Missing Values)
Trong y tế, một người đang sống không thể có chỉ số Glucose = 0 mg/dL hoặc BMI = 0 kg/m².
Do đó giá trị 0 ở hai cột này bản chất là dữ liệu bị thiếu.

In [ ]:
for col in ['Glucose', 'BMI', 'Age']:
    zeros = (df_sub[col] == 0).sum()
    pct = (zeros / len(df_sub)) * 100
    print(f'- {col}: {zeros} giá trị 0 ({pct:.2f}%)')

# Chuyển số 0 thành np.nan để phân tích thống kê chính xác
df_cleaned = df_sub.copy()
df_cleaned['Glucose'] = df_cleaned['Glucose'].replace(0, np.nan)
df_cleaned['BMI'] = df_cleaned['BMI'].replace(0, np.nan)

df_cleaned.describe()

### 4. Trực quan hóa dữ liệu (EDA)

In [ ]:
# 4.1. Phân bố nhãn Outcome
plt.figure(figsize=(6, 4), dpi=150)
sns.countplot(data=df_cleaned, x='Outcome', palette=['#3182ce', '#e53e3e'], hue='Outcome', legend=False)
plt.title('Phân bố nhãn mục tiêu Outcome')
plt.xticks([0, 1], ['Không tiểu đường (0)', 'Tiểu đường (1)'])
plt.show()

In [ ]:
# 4.2. Phân bố từng chỉ số theo tình trạng tiểu đường
fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=150)
for i, col in enumerate(features):
    sns.histplot(data=df_cleaned, x=col, hue='Outcome', kde=True, ax=axes[i], palette={0: '#3182ce', 1: '#e53e3e'}, alpha=0.4)
    axes[i].set_title(f'Phân bố {col}')
plt.tight_layout()
plt.show()

In [ ]:
# 4.3. Ma trận tương quan Pearson
plt.figure(figsize=(6, 5), dpi=150)
sns.heatmap(df_cleaned.corr(), annot=True, cmap='Blues', fmt='.2f', square=True)
plt.title('Ma trận tương quan Pearson')
plt.show()

### 5. Chiến lược chia dữ liệu và Tiền xử lý chống Data Leakage
> **Nguyên tắc vàng:** Tuyệt đối không tính toán trung vị hay mean/std trên toàn bộ tập dữ liệu trước khi chia train/test.
> Mọi bước học thông số (`fit`) chỉ được thực hiện trên tập Train.

In [ ]:
X = df_sub[features].copy()
y = df_sub[target].copy()

# Thay thế 0 bằng NaN
X['Glucose'] = X['Glucose'].replace(0, np.nan)
X['BMI'] = X['BMI'].replace(0, np.nan)

# Chia dữ liệu 80% train, 20% test có stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Tập huấn luyện (Train): {X_train.shape}')
print(f'Tập kiểm thử (Test):     {X_test.shape}')
print('\nTỷ lệ nhãn Train:\n', y_train.value_counts(normalize=True).round(4))
print('\nTỷ lệ nhãn Test:\n', y_test.value_counts(normalize=True).round(4))

In [ ]:
# Điền khuyết bằng Trung vị (Median) chỉ fit trên Train
imputer = SimpleImputer(strategy='median')
imputer.fit(X_train)

print('Trung vị học từ X_train:')
for col, val in zip(features, imputer.statistics_):
    print(f'- {col}: {val:.2f}')

X_train_imp = imputer.transform(X_train)
X_test_imp = imputer.transform(X_test)

# Chuẩn hóa thang đo (StandardScaler) chỉ fit trên Train
scaler = StandardScaler()
scaler.fit(X_train_imp)

X_train_scaled = scaler.transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

print('\nKiểm tra sau chuẩn hóa:')
print('Train mean (~0):', X_train_scaled.mean(axis=0).round(4))
print('Train std  (~1):', X_train_scaled.std(axis=0).round(4))
print('Test mean (khác 0):', X_test_scaled.mean(axis=0).round(4))
print('\nĐã hoàn thành tiền xử lý không rò rỉ dữ liệu!')